# DoubleIntegrator - Plot Evaluation Results

Load saved evaluation trajectories and select one to plot.


In [ ]:
import json
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "src" / "sdpc").is_dir() and (candidate / "case_studies").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the SD-DPC repository")

REPO_ROOT = find_repo_root()
CASE_DIR = REPO_ROOT / "case_studies" / "DoubleIntegrator"
sys.path.insert(0, str(REPO_ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
import torch

from sdpc.config import load_config
from sdpc.plotting import plot_controller_evaluation
from sdpc.registry import make_system

device = torch.device('cpu')
system = make_system('double_integrator', device=device)
CONFIGS = CASE_DIR / 'configs'
RESULTS = CASE_DIR / 'results'
cfg = load_config(CONFIGS / 'eval.yaml')
RESULT_DIR = RESULTS / 'eval' / 'controller_comparison'

payload = torch.load(RESULT_DIR / 'all_trajectories.pt', map_location=device)
trajectories = payload['trajectories']
scenarios = payload['scenarios']
metrics = json.loads((RESULT_DIR / 'metrics_per_seed.json').read_text())
print(f'Loaded {len(scenarios)} trajectories from {RESULT_DIR}')
print('Methods:', list(trajectories))


## Select trajectory

Set the index and rerun this cell.


In [ ]:
TRAJECTORY_INDEX = 0
n_trajectories = len(scenarios)
if not 0 <= TRAJECTORY_INDEX < n_trajectories:
    raise IndexError(f'TRAJECTORY_INDEX must be in [0, {n_trajectories - 1}]')

selected = {
    name: {
        'x_traj': values['x_traj'][TRAJECTORY_INDEX:TRAJECTORY_INDEX + 1],
        'u_traj': values['u_traj'][TRAJECTORY_INDEX:TRAJECTORY_INDEX + 1],
    }
    for name, values in trajectories.items()
}
fig, axes = plot_controller_evaluation(
    system, cfg, selected, scenarios[TRAJECTORY_INDEX]
)
figure_dir = RESULT_DIR / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)
figure_path = figure_dir / f'trajectory_{TRAJECTORY_INDEX}.pdf'
fig.savefig(figure_path, bbox_inches='tight')
plt.show()

display(pd.DataFrame({
    method: rows[TRAJECTORY_INDEX]
    for method, rows in metrics.items()
}).T)
print(f'Plotted trajectory {TRAJECTORY_INDEX} of {n_trajectories - 1}')
print('Saved:', figure_path)
